# Happy and sad faces from noise

A tiny diffusion model, small enough to train in seconds. Every picture is a face of $9 \times 9$ pixels, two eyes and a mouth, and the prompt is a single bit: $0$ asks for a happy face, $1$ for a sad one. The faces are drawn below in a few lines of Python, from a list of pixels.

The notebook follows the diffusion slides of Friday 11:00:

* the **forward process** blends a face into noise, in $T = 20$ steps,
* the **denoiser** looks at a noisy picture, the step $t$ and the bit, and estimates the picture one step earlier,
* **training** shows it noisy faces at random steps; the loss measures how far its estimate of the previous picture is from the true one,
* **generation** starts from pure noise and applies the denoiser $T$ times, from $t = T$ down to $t = 1$.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(0)   # seeded: the same noise and the same starting weights on every computer

## 1. The faces

A picture is a grid of $9 \times 9$ numbers, $0$ for empty paper and $1$ for ink. A face is two eyes and a mouth: the mouth curves up on a happy face and down on a sad one. Every face may sit one pixel higher or lower, and one pixel further left or right, which makes $9$ happy faces and $9$ sad ones. Each face comes with its bit: $0$ for happy, $1$ for sad.

In [ ]:
EYES = [(2, 3), (2, 5)]                                  # (row, column) of every ink pixel
HAPPY = [(5, 2), (6, 3), (6, 4), (6, 5), (5, 6)]         # the mouth curves up
SAD = [(6, 2), (5, 3), (5, 4), (5, 5), (6, 6)]           # the mouth curves down

faces, bits = [], []
for bit, mouth in [(0, HAPPY), (1, SAD)]:
    for dy in [-1, 0, 1]:                                # one pixel up, none, one pixel down
        for dx in [-1, 0, 1]:                            # one pixel left, none, one pixel right
            face = torch.zeros(9, 9)
            for row, col in EYES + mouth:
                face[row + dy, col + dx] = 1.0
            faces.append(face.flatten())                 # the 81 pixels in one row
            bits.append([bit])
faces = torch.stack(faces)                               # 18 faces, 81 numbers each
bits = torch.tensor(bits, dtype=torch.float32)           # 0 for happy, 1 for sad
print(faces.shape, bits.shape)

In [ ]:
def show(ax, picture, title=""):
    ax.imshow(picture.reshape(9, 9), cmap="Greys", vmin=0, vmax=1)
    ax.set(xticks=[], yticks=[], title=title)


fig, axes = plt.subplots(2, 9, figsize=(13, 3.4))
for ax, face, bit in zip(axes.flat, faces, bits):
    show(ax, face, "sad" if bit.item() == 1 else "happy")
plt.show()

## 2. The forward process

At step $t$ the picture is a blend of the face and a picture of noise: $1 - t/T$ of the face plus $t/T$ of the noise. At $t = 0$ it is the face itself, at $t = T$ pure noise. On the slides every step adds fresh noise; here the face fades into one noise picture, a little more at every step, which looks the same and keeps the code short.

In [ ]:
T = 20                                                   # the number of steps from a face to pure noise


def add_noise(face, noise, t):
    return (1 - t / T) * face + (t / T) * noise         # the picture at step t

In [ ]:
noise = torch.randn(81)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for ax, t in zip(axes, [0, 4, 8, 12, 16, 20]):
    show(ax, add_noise(faces[4], noise, t), f"t = {t}")
plt.show()

## 3. The denoiser

The network reads $81 + 1 + 1 = 83$ numbers: the noisy picture $x_t$, the step $t$ (as $t/T$) and the bit. It answers with $81$ numbers, its guess of the clean face. `previous_picture` then takes one step from $x_t$ towards that guess, a $1/t$ part of the way, and that is the denoiser's estimate of $x_{t-1}$. At $t = 1$ the step goes all the way, so the estimate of $x_0$ is the guess itself.

On the slides the network guesses the noise. Here it guesses the clean face. Given the noisy picture, either guess gives the other.

In [ ]:
net = nn.Sequential(
    nn.Linear(81 + 1 + 1, 128), nn.ReLU(),               # in: the noisy picture, the step and the bit
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 81),                                  # out: its guess of the clean face, one number per pixel
)


def previous_picture(x, t, bit):
    guess = net(torch.cat([x, t / T, bit], dim=1))       # the network's guess of the clean face
    return x + (guess - x) / t                           # one step from x_t towards it: the estimate of x_{t-1}

## 4. Training

The loop of weekend 1: while the loss is too big, take a step. Every round picks $256$ faces at random, each with its own noise picture and its own step $t$ between $1$ and $T$. For each face, the loss sums over its $81$ pixels the squared difference between the denoiser's estimate of the previous picture and the true previous picture: the same face, blended with the same noise, at step $t - 1$. Then it averages over the $256$ faces, as $\bar{\ell}$ on the slides. Training takes well under a minute.

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

loss = 1.0                                                       # any start above the threshold
while loss > 0.012:
    pick = torch.randint(0, 18, (256,))                          # 256 faces picked at random
    face, bit = faces[pick], bits[pick]
    noise = torch.randn_like(face)                               # a noise picture for each face
    t = torch.randint(1, T + 1, (256, 1)).float()                # a step between 1 and T for each face
    x_t = add_noise(face, noise, t)                              # the noisy picture at step t
    x_previous = add_noise(face, noise, t - 1)                   # the true picture one step earlier
    loss = ((previous_picture(x_t, t, bit) - x_previous) ** 2).sum(dim=1).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"\rloss: {loss.item():.4f}", end="")

## 5. Generating faces

The three steps of the slides: (1) sample noise, (2) for $t = T, T-1, \dots, 1$, estimate $x_{t-1}$ from $x_t$, (3) output $x_0$. Every face starts from its own noise, so the same bit gives different faces.

In [ ]:
def generate(bit, how_many=10):
    with torch.no_grad():
        x = torch.randn(how_many, 81)                            # 1) sample noise: x_T
        bit_column = torch.full((how_many, 1), float(bit))
        t = T
        while t > 0:                                             # 2) for t = T, T-1, ..., 1:
            step = torch.full((how_many, 1), float(t))
            x = previous_picture(x, step, bit_column)            #    estimate x_{t-1} from x_t
            t = t - 1
    return x                                                     # 3) output x_0

In [ ]:
happy, sad = generate(0), generate(1)
fig, axes = plt.subplots(2, 10, figsize=(15, 3.4))
for ax, picture in zip(axes[0], happy):
    show(ax, picture, "bit 0")
for ax, picture in zip(axes[1], sad):
    show(ax, picture, "bit 1")
plt.show()

## 6. One face, step by step

The same steps for one happy face, every other picture shown: pure noise on the left, $x_0$ on the right. The face comes through in the last steps.

In [ ]:
with torch.no_grad():
    x = torch.randn(1, 81)
    pictures = [x]
    t = T
    while t > 0:
        x = previous_picture(x, torch.full((1, 1), float(t)), torch.full((1, 1), 0.0))
        pictures.append(x)
        t = t - 1
fig, axes = plt.subplots(1, 11, figsize=(16, 2))
for ax, picture, t in zip(axes, pictures[::2], [20, 18, 16, 14, 12, 10, 8, 6, 4, 2, 0]):
    show(ax, picture, f"t = {t}")
plt.show()

## 7. Try it

* Run the last cells again: new noise, new faces.
* The bit is a number, so give it one the network never saw. The cell below asks for bit $0.5$: which mood comes out?
* Make the training threshold bigger, say $0.02$, run the denoiser cell again (it starts the network from scratch), train, and compare the faces.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
for ax, picture in zip(axes, generate(0.5)):
    show(ax, picture, "bit 0.5")
plt.show()